Classify if Demented, Nondemented

In [ ]:
#set up a ColumnTransformer with StandardScaler for numerical features and OneHotEncoder for categorical features.
#set up and training a LinearRegression model using scikit-learn, including data preprocessing steps within a Pipeline.
#implement polynomial regression
#perform hyperparameter tuning for a polynomial regression model
#evaluate the performance of a regression model on test data
#use OneHotEncoder with handle_unknown='ignore' within a preprocessing pipeline to handle unseen categories during model training and evaluation
#set up and execute cross_val_score or GridSearchCV to perform cross-validation

In [1]:
from google.colab import files
uploaded = files.upload()


Saving dementia.csv to dementia.csv


In [13]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split

In [5]:
df = pd.read_csv("dementia.csv")
print(df.head())


  Subject ID         MRI ID        Group  Visit  MR Delay M/F Hand  Age  EDUC  \
0  OAS2_0001  OAS2_0001_MR1  Nondemented      1         0   M    R   87    14   
1  OAS2_0001  OAS2_0001_MR2  Nondemented      2       457   M    R   88    14   
2  OAS2_0002  OAS2_0002_MR1     Demented      1         0   M    R   75    12   
3  OAS2_0002  OAS2_0002_MR2     Demented      2       560   M    R   76    12   
4  OAS2_0002  OAS2_0002_MR3     Demented      3      1895   M    R   80    12   

   SES  MMSE  CDR  eTIV   nWBV    ASF  
0  2.0  27.0  0.0  1987  0.696  0.883  
1  2.0  30.0  0.0  2004  0.681  0.876  
2  NaN  23.0  0.5  1678  0.736  1.046  
3  NaN  28.0  0.5  1738  0.713  1.010  
4  NaN  22.0  0.5  1698  0.701  1.034  


In [6]:
print(df.isnull().sum())


Subject ID     0
MRI ID         0
Group          0
Visit          0
MR Delay       0
M/F            0
Hand           0
Age            0
EDUC           0
SES           19
MMSE           2
CDR            0
eTIV           0
nWBV           0
ASF            0
dtype: int64


In [8]:
print(df.columns)

Index(['Subject ID', 'MRI ID', 'Group', 'Visit', 'MR Delay', 'M/F', 'Hand',
       'Age', 'EDUC', 'SES', 'MMSE', 'CDR', 'eTIV', 'nWBV', 'ASF'],
      dtype='object')


In [11]:
# Identify features and target
target_col = "Group"  # Classification target (Demented / Nondemented)

# Convert target labels to numerical values
df[target_col] = df[target_col].map({'Demented': 1, 'Nondemented': 0})

# Identify categorical and numerical features
categorical_features = ['M/F', 'Hand']  # Exclude 'Group' as it's the target
numerical_features = ['MR Delay', 'Age', 'EDUC', 'SES', 'MMSE', 'CDR', 'eTIV', 'nWBV', 'ASF']

# Handle missing values (Optional)
df.fillna(df.median(numeric_only=True), inplace=True)
df.fillna(df.mode().iloc[0], inplace=True)

# Define ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

# Splitting into features (X) and target (y)
X = df.drop(columns=[target_col])
y = df[target_col]

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit and transform the data
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

# Check transformed shape
print("Transformed X_train shape:", X_train_transformed.shape)
print("Transformed X_test shape:", X_test_transformed.shape)



Transformed X_train shape: (298, 12)
Transformed X_test shape: (75, 12)


In [15]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

model_pipeline.fit(X_train, y_train)

y_pred = model_pipeline.predict(X_test)



In [23]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

degree = 2
model_pipeline = make_pipeline(
    preprocessor,
    PolynomialFeatures(degree),
    LinearRegression()
)

model_pipeline.fit(X_train, y_train)

y_pred = model_pipeline.predict(X_test)


In [17]:
from sklearn.model_selection import GridSearchCV

param_grid = {'polynomialfeatures__degree': [2, 3, 4, 5]}

model_pipeline = make_pipeline(
    preprocessor,
    PolynomialFeatures(),
    LinearRegression()
)

grid_search = GridSearchCV(model_pipeline, param_grid, cv=5, scoring='r2')
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
best_degree = grid_search.best_params_['polynomialfeatures__degree']

y_pred = best_model.predict(X_test)

print(f"Best polynomial degree: {best_degree}")


Best polynomial degree: 2


In [18]:
from sklearn.metrics import mean_squared_error, r2_score

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.4f}")
print(f"R² Score: {r2:.4f}")


Mean Squared Error: 0.0446
R² Score: 0.8175


In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])


In [20]:
from sklearn.model_selection import cross_val_score, GridSearchCV

param_grid = {'polynomialfeatures__degree': [2, 3, 4, 5]}

model_pipeline = make_pipeline(
    preprocessor,
    PolynomialFeatures(),
    LinearRegression()
)

grid_search = GridSearchCV(model_pipeline, param_grid, cv=5, scoring='r2')
grid_search.fit(X_train, y_train)

cv_scores = cross_val_score(grid_search.best_estimator_, X_train, y_train, cv=5, scoring='r2')

print(f"Cross-validation scores: {cv_scores}")
print(f"Mean R² Score: {cv_scores.mean():.4f}")


Cross-validation scores: [0.59134477 0.76416615 0.70071309 0.70001294 0.81934992]
Mean R² Score: 0.7151


In [21]:

print("Best score:", grid_search.best_score_)


Best score: 0.715117374016011
